CSC 702 Mathematics of AI  
Group Project 4  
Tokenization
 


In [19]:
import numpy as np
import pandas as pd
import fasttext
import sklearn
import torch
import re
import sentencepiece as spm

In [20]:
# I copy and pasted some data from Project Gutenberg, this is my attempt to split it into a csv style file with every sentence on it's own line
def process_corpus_with_sentences(input_file, output_file):
    with open(input_file, 'r', encoding='utf-8') as f:
        text = f.read()
        sentences = re.split(r'(?<=[.!?])\s+', text)

    with open(output_file, 'w', encoding='utf-8') as f:
        for sentence in sentences:
            tokens = sentence.strip().lower().split()
            if tokens: 
                f.write(" ".join(tokens) + "\n")

    print(f'Processed corpus written to {output_file}')    

In [21]:
process_corpus_with_sentences('corpus.txt', 'cleaned_corpus.txt')

Processed corpus written to cleaned_corpus.txt


In [22]:
model = fasttext.train_unsupervised('cleaned_corpus.txt', model='skipgram', dim = 50)

Read 0M words
Number of words:  8230
Number of labels: 0
Progress: 100.0% words/sec/thread:  423491 lr:  0.000000 avg.loss:  2.453122 ETA:   0h 0m 0s


In [23]:
print(model.get_word_vector('she'))
print(model.get_nearest_neighbors('she'))

[ 0.12633668  0.17465423 -1.168014   -0.38383526 -0.13626537  0.3587211
 -0.17641166 -0.05553313  0.6068783   0.87001306  0.47567728  0.55400336
  0.583266    0.09396448  0.26177284  0.5848883  -0.42217258 -0.55602163
  0.09995566  0.24277146  0.29827675  0.7144677   0.20080797  0.12530744
 -0.1202805   0.06220181 -0.13147718 -0.19585392 -0.11912275 -0.57370925
  0.26429397 -0.4504286   0.01779357 -0.07761659 -0.24298999 -0.14001366
  0.4400314   0.58291686  0.5676484   0.2993764  -0.30396518  0.53956956
 -0.42258403 -0.4569212   0.41283056 -0.7455218  -0.20999971 -0.28914365
  0.16207765 -0.1269755 ]
[(0.9278514981269836, 'it;'), (0.9202978014945984, '(she'), (0.9101930260658264, 'neatly'), (0.9101055264472961, 'pleasantly'), (0.9081573486328125, 'almost'), (0.9047072529792786, '_very_'), (0.9018539190292358, 'lonely'), (0.9003924131393433, 'herself;'), (0.8989547491073608, 'first,'), (0.8977229595184326, 'justly')]


In [24]:
spm.SentencePieceTrainer.Train(
    input = 'cleaned_corpus.txt',
    model_prefix='bpe',
    vocab_size = 10000,
    model_type = 'bpe',
)

# Load Tokenizer
sp = spm.SentencePieceProcessor()
sp.Load('bpe.model')

sentencepiece_trainer.cc(78) LOG(INFO) Starts training with : 
trainer_spec {
  input: cleaned_corpus.txt
  input_format: 
  model_prefix: bpe
  model_type: BPE
  vocab_size: 10000
  self_test_sample_size: 0
  character_coverage: 0.9995
  input_sentence_size: 0
  shuffle_input_sentence: 1
  seed_sentencepiece_size: 1000000
  shrinking_factor: 0.75
  max_sentence_length: 4192
  num_threads: 16
  num_sub_iterations: 2
  max_sentencepiece_length: 16
  split_by_unicode_script: 1
  split_by_number: 1
  split_by_whitespace: 1
  split_digits: 0
  pretokenization_delimiter: 
  treat_whitespace_as_suffix: 0
  allow_whitespace_only_pieces: 0
  required_chars: 
  byte_fallback: 0
  vocabulary_output_piece_score: 1
  train_extremely_large_corpus: 0
  seed_sentencepieces_file: 
  hard_vocab_limit: 1
  use_all_vocab: 0
  unk_id: 0
  bos_id: 1
  eos_id: 2
  pad_id: -1
  unk_piece: <unk>
  bos_piece: <s>
  eos_piece: </s>
  pad_piece: <pad>
  unk_surface:  ⁇ 
  enable_differential_privacy: 0
  differe

True

_trainer.cc(268) LOG(INFO) Added: freq=14 size=5740 all=29309 active=1570 piece=▁ade
bpe_model_trainer.cc(268) LOG(INFO) Added: freq=14 size=5760 all=29364 active=1625 piece=haust
bpe_model_trainer.cc(268) LOG(INFO) Added: freq=14 size=5780 all=29371 active=1632 piece=▁rent
bpe_model_trainer.cc(268) LOG(INFO) Added: freq=14 size=5800 all=29401 active=1662 piece=▁blank
bpe_model_trainer.cc(159) LOG(INFO) Updating active symbols. max_freq=14 min_freq=10
bpe_model_trainer.cc(268) LOG(INFO) Added: freq=14 size=5820 all=29404 active=1473 piece=▁range
bpe_model_trainer.cc(268) LOG(INFO) Added: freq=14 size=5840 all=29395 active=1464 piece=▁impart
bpe_model_trainer.cc(268) LOG(INFO) Added: freq=14 size=5860 all=29384 active=1453 piece=▁tumult
bpe_model_trainer.cc(268) LOG(INFO) Added: freq=14 size=5880 all=29385 active=1454 piece=▁grieved
bpe_model_trainer.cc(268) LOG(INFO) Added: freq=14 size=5900 all=29369 active=1438 piece=▁thunder
bpe_model_trainer.cc(159) LOG(INFO) Updating active symbol

In [25]:
with open("cleaned_corpus.txt", 'r', encoding='utf-8') as fin, \
    open ("bpe_corpus.txt", 'w', encoding='utf-8') as fout:
    for line in fin:
        line = line.strip()
        if not line:
            continue
        tokens = sp.EncodeAsPieces(line)
        fout.write(' '.join(tokens) + '\n')

print("BPE corpus written to bpe_corpus.txt")

BPE corpus written to bpe_corpus.txt


In [26]:
model_byte_pair = fasttext.train_unsupervised('bpe_corpus.txt', model='skipgram', dim = 50)

Read 0M words
Number of words:  8633
Number of labels: 0
Progress: 100.0% words/sec/thread:  358191 lr:  0.000000 avg.loss:  2.462449 ETA:   0h 0m 0s


In [27]:
print(model_byte_pair.get_word_vector('she'))
print(model_byte_pair.get_nearest_neighbors('she'))

[-0.5216937   0.35348076  0.36376885  0.41392076 -0.23075797 -0.4795979
  0.8144019   0.05225019  0.32167763 -0.33169338 -0.1857886   0.07313804
 -0.13815814 -0.02036772  0.25834766 -0.39148343 -0.23397784  0.4009554
 -0.21434765  0.11572876 -0.3411316   0.82486254 -0.08713476 -0.29681724
 -0.21040508 -0.18071164  0.5719821  -0.2563367   0.6747701   0.84871274
  0.11264011 -0.01725928 -0.75578344 -0.40356353  0.21138842 -0.16776858
 -0.15355203  0.45493272  0.24831548  0.06321338  0.8659814   0.7577172
  0.38146558  0.18869786 -0.06499787  0.00515452 -0.8348684   0.34305486
  0.02416604  0.2600255 ]
[(0.8190730214118958, 'whe'), (0.8154114484786987, 'never'), (0.8149575591087341, 'he'), (0.799322783946991, 'alice'), (0.7864152789115906, 'when'), (0.772952139377594, 'ever'), (0.7726468443870544, 'some'), (0.7714670300483704, '“'), (0.764435887336731, '▁beethoven'), (0.760732889175415, 'which')]
